# 1. Pretraining Loss

In [ ]:
import re
import matplotlib.pyplot as plt

def extract_validation_losses(log_file_path):
    """
    Extracts Validation loss 1 and 2 from the given log file.
    """
    pattern = re.compile(
        r"Train loss, Validation loss, Validation loss 2, .*: [\d.]+, ([\d.]+), ([\d.]+)"
    )
    val1, val2 = [], []

    with open(log_file_path, "r") as f:
        for line in f:
            match = pattern.search(line)
            if match:
                val1.append(float(match.group(1)))
                val2.append(float(match.group(2)))
    return val1, val2


def plot_split_validation_losses(log_paths, labels, save_path=None):
    """
    Plots Validation Loss 1 and Validation Loss 2 from multiple training modes
    in two side-by-side subplots for compact article-style display.
    Optionally saves the figure to disk with high DPI.
    """
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)

    colors = ['tab:blue', 'tab:orange', 'tab:green']  # consistent color scheme

    # --- Validation Loss 1 ---
    for path, label, color in zip(log_paths, labels, colors):
        val1, val2 = extract_validation_losses(path)
        if not val1:
            print(f"⚠️ No validation losses found in {path}")
            continue
        epochs = range(1, len(val1) + 1)
        axes[0].plot(epochs, val1, label=label, color=color, linewidth=1.8)
    axes[0].set_title("Validation Loss 1")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=9)

    # --- Validation Loss 2 ---
    for path, label, color in zip(log_paths, labels, colors):
        val1, val2 = extract_validation_losses(path)
        if not val2:
            continue
        epochs = range(1, len(val2) + 1)
        axes[1].plot(epochs, val2, label=label, color=color, linewidth=1.8)
    axes[1].set_title("Validation Loss 2")
    axes[1].set_xlabel("Epoch")

    plt.tight_layout()

    # --- Save high-DPI figure ---
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight')  # 300 DPI for print-quality
        print(f"✅ Figure saved to: {save_path}")

    plt.show()


# Example usage
plot_split_validation_losses(
    [
        "./checkpoint/static/train_model.log",
        "./checkpoint/dynamic_mask/train_model.log",
        "./checkpoint/dynamic_mask_shift/train_model.log"
    ],
    ["Static", "Dynamic Mask", "Dynamic Mask + CHUCKLES Shifting"],
    save_path="./manuscript/figure/pretraining_loss.png"  # <- save here
)

# 2. Masking Percentage

In [ ]:
import random 
import matplotlib.pyplot as plt
import numpy as np

def get_num_mask(chuckles: str):
    monomers = chuckles.split('|')
    return round(random.triangular(1, len(monomers)*0.4, 0))

with open("./data/train_data/train_filled_chuckles.txt", "r") as file:
    chuckles_data = [x.strip() for x in file.readlines()]

In [ ]:
num_masks = [get_num_mask(x) for x in chuckles_data]
# Compute frequencies of each unique value
unique, counts = np.unique(num_masks, return_counts=True)

plt.figure(figsize=(6, 4))
plt.bar(unique, counts, color='skyblue', edgecolor='black')
plt.xlabel("Number of Masks")
plt.ylabel("Frequency")
plt.xticks(unique)
# plt.savefig("./figure_pdf/mask_distribution.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

pepinvent_train = pd.read_csv('./data/train_data/original/train.csv')
pepinvent_val = pd.read_csv('./data/train_data/original/val.csv')

pepinvent = pd.concat([pepinvent_train, pepinvent_val], ignore_index=True)
pepinvent['num_mask'] = pepinvent['Target_Mol'].apply(lambda x: len(x.split('|')))

plt.figure(figsize=(6, 4))
plt.bar(pepinvent['num_mask'].value_counts().index, pepinvent['num_mask'].value_counts().values, color='salmon', edgecolor='black')
plt.xlabel("Number of Masks")
plt.ylabel("Frequency")
plt.title("Distribution of Number of Masks in PepInvent Dataset")
# plt.savefig("./figure_pdf/pepinvent_mask_distribution.png", dpi=600, bbox_inches='tight')

In [ ]:
# Distribution of Number of Masks: PepEVOLVE vs PepINVENT
unique_pepevolve, counts_pepevolve = np.unique(num_masks, return_counts=True)
unique_pepinvent, counts_pepinvent = np.unique(pepinvent['num_mask'], return_counts=True)

x = np.arange(1, max(unique_pepevolve.max(), unique_pepinvent.max()) + 1)
width = 0.4

fig, ax = plt.subplots(figsize=(6, 4))

bars1 = ax.bar(x - width/2, 
               [counts_pepevolve[unique_pepevolve == i][0] if i in unique_pepevolve else 0 for i in x],
               width, label='PepEVOLVE', color='skyblue', edgecolor='black')

bars2 = ax.bar(x + width/2, 
               [counts_pepinvent[unique_pepinvent == i][0] if i in unique_pepinvent else 0 for i in x],
               width, label='PepINVENT', color='salmon', edgecolor='black')

ax.set_xlabel("Number of Masks")
ax.set_ylabel("Frequency")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.savefig("./figure_pdf/mask_distribution_comparison.pdf", dpi=600, bbox_inches='tight')
plt.show()

# 3. Router policy distribution

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

def plot_categorical_evolution(distributions, save_path=None):
    # Convert list of arrays into 2D NumPy array (T × C)
    data = np.vstack(distributions)
    timesteps, categories = data.shape

    # plt.figure(figsize=(max(6, categories/2), max(3, timesteps/100)), dpi=600)
    plt.figure(figsize=(5, 4), dpi=600)
    im = plt.imshow(
        data,
        aspect='auto',
        interpolation='bilinear',  # or 'bicubic' for even smoother
        origin='upper',
        vmin=0.0,
        vmax=1.0
    )
    plt.colorbar(im, label='Probability')

    # plt.title(f'Router Optimal Position Distribution')
    plt.xlabel('Positions')
    plt.ylabel('Steps')

    # Reduce tick clutter for large numbers
    plt.xticks(range(categories))
    yticks = np.arange(0, timesteps, 50)
    plt.yticks(ticks=yticks, labels=[str(i) for i in yticks])

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=600)
    else:
        plt.show()
    plt.close()

In [ ]:
hbd_1pos = np.load('./output/logging/hbd_1pos/router_ps.npy', allow_pickle=True)
hbd_2pos = np.load('./output/logging/hbd_2pos/router_ps.npy', allow_pickle=True)

logp_luna18_1pos = np.load('./output/logging/logp_luna18_1pos/router_ps.npy', allow_pickle=True)
logp_luna18_2pos = np.load('./output/logging/logp_luna18_2pos/router_ps.npy', allow_pickle=True)

logp_luna18_2pos_adversarial = np.load('./output/logging/logp_luna18_2pos_adversarial/router_ps.npy', allow_pickle=True)

In [ ]:
plot_categorical_evolution(hbd_1pos, save_path='./manuscript/figure/hbd_1pos.svg')
plot_categorical_evolution(hbd_2pos, save_path='./manuscript/figure/hbd_2pos.svg')

In [ ]:
plot_categorical_evolution(logp_luna18_1pos, save_path='./manuscript/figure/logp_luna18_1pos.svg')
plot_categorical_evolution(logp_luna18_2pos, save_path='./manuscript/figure/logp_luna18_2pos.svg')

In [ ]:
plot_categorical_evolution(logp_luna18_2pos_adversarial, save_path='./manuscript/figure/logp_luna18_2pos_adversarial.svg')

# 4. PepEVOLVE vs PepINVENT

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 

In [ ]:
pepinvent = pd.read_csv('./output/logging/crbp_pepinvent/results.csv')

neighbor_multi = pd.read_csv('./output/result/crbp_neighbor_multi/results_evolving_step250.csv')
neighbor_single = pd.read_csv('./output/result/crbp_neighbor_single/results_evolving_step250.csv')
self_multi = pd.read_csv('./output/result/crbp_self_multi/results_evolving_step250.csv')
self_single = pd.read_csv('./output/result/crbp_self_single/results_evolving_step250.csv')

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",  # SVG: keep fonts as text
    "pdf.fonttype": 42,      # PDF: TrueType fonts (editable in Illustrator)
    "ps.fonttype": 42        # EPS/PS if you ever use it
})

mean_self_multi = self_multi.groupby(self_multi['Step'].astype(int))['total_score'].mean() 
mean_self_single = self_single.groupby(self_single['Step'].astype(int))['total_score'].mean() 
mean_neighbor_multi = neighbor_multi.groupby(neighbor_multi['Step'].astype(int))['total_score'].mean() 
mean_neighbor_single = neighbor_single.groupby(neighbor_single['Step'].astype(int))['total_score'].mean() 

mean_pepinvent = pepinvent.groupby(pepinvent['Step'].astype(int))['total_score'].mean() 


# combine into one DataFrame (index = Step)
mean_scores_by_step = pd.DataFrame({
    'PepINVENT': mean_pepinvent,
    'PepEVOLVE - NM': mean_neighbor_multi,
    'PepEVOLVE - NS': mean_neighbor_single,
    'PepEVOLVE - SM': mean_self_multi,
    'PepEVOLVE - SS': mean_self_single,
}).sort_index()

# Plot time evolution (Step) of summed scores (divided by 128)
plt.figure(figsize=(5, 4))
colors = ['tab:gray', 'tab:blue', 'tab:green', 'tab:purple', 'tab:red']

for col, c in zip(mean_scores_by_step.columns, colors):
    plt.plot(mean_scores_by_step.index, mean_scores_by_step[col], label=col, color=c, linewidth=1.6, alpha=0.8)

plt.xlabel('Step')
plt.ylabel('Score')
plt.legend(fontsize=9)
plt.tight_layout()

# Save as vector (SVG or PDF). DPI is irrelevant for vector output, so you can omit it.
outbase = "./manuscript/figure/score_linegraph_raw"

plt.savefig(f"{outbase}.svg", bbox_inches="tight")   # SVG (great for web / editing)
# or:
plt.savefig(f"{outbase}.pdf", bbox_inches="tight")   # PDF (great for LaTeX / print)

plt.show()

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",  # SVG: keep fonts as text
    "pdf.fonttype": 42,      # PDF: TrueType fonts (editable in Illustrator)
    "ps.fonttype": 42        # EPS/PS if you ever use it
})

# recompute sum total_score per integer Step for each dataset and divide by 128
mean_neighbor_multi = neighbor_multi.groupby(neighbor_multi['Step'].astype(int))['total_score'].sum() / 128
mean_neighbor_single = neighbor_single.groupby(neighbor_single['Step'].astype(int))['total_score'].sum() / 128
mean_pepinvent = pepinvent.groupby(pepinvent['Step'].astype(int))['total_score'].sum() / 128

# combine into one DataFrame (index = Step)
mean_scores_by_step = pd.DataFrame({
    'PepINVENT': mean_pepinvent,
    'PepEVOLVE - NM': mean_neighbor_multi,
    'PepEVOLVE - NS': mean_neighbor_single,
}).sort_index()

# Plot time evolution (Step) of summed scores (divided by 128)
plt.figure(figsize=(5, 4))
colors = ['tab:gray', 'tab:blue', 'tab:green']

for col, c in zip(mean_scores_by_step.columns, colors):
    plt.plot(mean_scores_by_step.index, mean_scores_by_step[col], label=col, color=c, linewidth=1.6, alpha=0.9)

plt.xlabel('Step')
plt.ylabel('Score')
plt.legend(fontsize=9)
plt.tight_layout()

plt.savefig("./manuscript/figure/score_linegraph_neighbor_mask.svg", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",  # SVG: keep fonts as text
    "pdf.fonttype": 42,      # PDF: TrueType fonts (editable in Illustrator)
    "ps.fonttype": 42        # EPS/PS if you ever use it
})

# --- existing aggregation ---
mean_self_multi = self_multi.groupby(self_multi['Step'].astype(int))['total_score'].sum() / 128
mean_self_single = self_single.groupby(self_single['Step'].astype(int))['total_score'].sum() / 128
mean_pepinvent = pepinvent.groupby(pepinvent['Step'].astype(int))['total_score'].sum() / 128

mean_scores_by_step = pd.DataFrame({
    'PepINVENT': mean_pepinvent,
    'PepEVOLVE - SM': mean_self_multi,
    'PepEVOLVE - SS': mean_self_single
}).sort_index()

# --- TensorBoard-style smoothing ---
def tensorboard_smooth(series: pd.Series, smoothing: float = 0.6) -> pd.Series:
    if not (0 <= smoothing < 1):
        raise ValueError("smoothing must be in [0, 1).")
    y = series.astype(float).copy()
    out = np.empty(len(y), dtype=float)
    last = np.nan
    alpha = smoothing
    beta = 1.0 - alpha

    for i, v in enumerate(y.values):
        if np.isnan(v):
            out[i] = last
        else:
            last = v if np.isnan(last) else (alpha * last + beta * v)
            out[i] = last
    return pd.Series(out, index=y.index)

SMOOTHING = 0.8
smoothed_scores = mean_scores_by_step.apply(tensorboard_smooth, smoothing=SMOOTHING)

# --- Plot without smoothed legend ---
fig, ax = plt.subplots(figsize=(5, 4))
colors = ['tab:gray', 'tab:purple', 'tab:red']

# plot raw (faint, labeled)
for col, c in zip(mean_scores_by_step.columns, colors):
    ax.plot(mean_scores_by_step.index, mean_scores_by_step[col],
            label=col, color=c, linewidth=0.7, alpha=0.3)

# plot smoothed (bold, unlabeled)
for col, c in zip(smoothed_scores.columns, colors):
    ax.plot(smoothed_scores.index, smoothed_scores[col],
            color=c, linewidth=1.0, alpha=0.95)

ax.set_xlabel('Step')
ax.set_ylabel('Score')
legend = ax.legend(fontsize=9, loc='lower left')

# Force legend lines to use raw color (alpha=1.0)
for line in legend.get_lines():
    line.set_alpha(1.0)
    line.set_linewidth(1.5)  # Adjust this value to your preference

plt.tight_layout()
plt.savefig("./manuscript/figure/score_linegraph_self_mask.svg",
            dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",  # SVG: keep fonts as text
    "pdf.fonttype": 42,      # PDF: TrueType fonts (editable in Illustrator)
    "ps.fonttype": 42        # EPS/PS if you ever use it
})

plt.figure(figsize=(6, 4))
plt.hist(neighbor_multi['total_score'], bins=100, alpha=0.9, label='PepEVOLVE - NM', color='tab:blue')
plt.hist(neighbor_single['total_score'], bins=100, alpha=0.5, label='PepEVOLVE - NS', color='green')
plt.hist(self_multi['total_score'], bins=100, alpha=0.5, label='PepEVOLVE - SM', color='purple')
plt.hist(self_single['total_score'], bins=100, alpha=0.5, label='PepEVOLVE - SS', color='red')
plt.hist(pepinvent['total_score'], bins=100, alpha=0.5, label='PepINVENT', color='gray')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend()
plt.savefig("./manuscript/figure/score_distribution.svg", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",  # SVG: keep fonts as text
    "pdf.fonttype": 42,      # PDF: TrueType fonts (editable in Illustrator)
    "ps.fonttype": 42        # EPS/PS if you ever use it
})

# Bins and labels
bins = np.array([0.4, 0.6, 0.8, 0.9, 0.94, 1.0])
labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)]
n_bins = len(labels)

# Histograms per dataset
pepinvent_hist, _   = np.histogram(pepinvent['total_score'],  bins=bins)
self_multi_hist, _  = np.histogram(self_multi['total_score'], bins=bins)
self_single_hist, _ = np.histogram(self_single['total_score'], bins=bins)
neighbor_single_hist, _ = np.histogram(neighbor_single['total_score'], bins=bins)
neighbor_multi_hist, _ = np.histogram(neighbor_multi['total_score'], bins=bins)

# Stack the data
data = np.vstack([
    pepinvent_hist,
    self_multi_hist,
    self_single_hist,
    neighbor_single_hist,
    neighbor_multi_hist
])

# Elegant color palette (muted blues/greens/grays)
colors_stack = ['#4C72B0', '#55A868', '#C44E52', '#8172B3', '#64B5CD', '#64B5CE']
colors_stack = ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#a2fab1']


datasets = ['PepINVENT', 'PepEVOLVE - SM', 'PepEVOLVE - SS', 'PepEVOLVE - NS', 'PepEVOLVE - NM']
x_pos = np.arange(len(datasets))
bottom = np.zeros(len(datasets), dtype=int)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))

for i in range(n_bins):
    ax.bar(
        x_pos, data[:, i],
        bottom=bottom,
        label=labels[i],
        color=colors_stack[i],
        edgecolor='black',
        linewidth=0.4
    )
    bottom += data[:, i]

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('Number of Unique Peptides', fontsize=11)
ax.set_xticks(x_pos, datasets, rotation=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Move legend outside the plot
ax.legend(title='Score Range', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)

plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave room for the legend
plt.savefig(
    "./manuscript/figure/score_stacked_bar.svg",
    dpi=600, bbox_inches='tight'
)
plt.show()

# 5. Ablation

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

base_dir = "./output/journal/result"

SETTINGS = ['nm', 'ns', 'sm', 'ss']
SETTING_TITLES = {
    'nm': 'Neighbor-Mask Multi-Agents (NM)',
    'ns': 'Neighbor-Mask Single-Agent (NS)',
    'sm': 'Self-Mask Multi-Agents (SM)',
    'ss': 'Self-Mask Single-Agent (SS)'
}

ABLATION_LABEL_ORDER = ['pepevolve', 'cs_topk', 'topk_gra', 'topk', 'pepinvent_cs', 'pepinvent']

ABLATION_LABELS = {
    'pepevolve': 'PepEVOLVE (CHUCKLES Shift + TopK + GRA)',
    'cs_topk':   'CHUCKLES Shift + TopK',
    'topk_gra':  'TopK + GRA',
    'topk':      'TopK',
    'pepinvent_cs': 'CHUCKLES Shift',
    'pepinvent': 'PepINVENT'
}

ABLATION_COLORS = {
    'pepevolve': "#0072B2",
    'cs_topk':   "#E69F00",
    'topk':      "#009E73",
    'topk_gra':  "#CC79A7",
    'pepinvent_cs': "#D55E00",
    'pepinvent': "#4C4C4C"
}

def get_ablation_type(mode, setting):
    """Extract ablation type from mode name."""
    if mode.startswith("pepinvent"):
        return mode
    prefix = f"{setting}_"
    if mode.startswith(prefix):
        return mode[len(prefix):]
    return mode

def load_runs(base_dir, mode):
    dfs = []
    for run in [1, 2, 3]:
        folder = os.path.join(base_dir, f"{mode}{run}")
        if mode.startswith("pepinvent"):
            csv_path = os.path.join(folder, "results.csv")
        else:
            csv_path = os.path.join(folder, "results_evolving_step250.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            dfs.append(df)
        else:
            print(f"  ⚠️ Missing: {csv_path}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

legend_handles = {}

for ax, setting in zip(axes.flat, SETTINGS):
    all_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
    all_folders = [f for f in all_folders if f.startswith(setting + "_") or f.startswith("pepinvent")]

    mode_set = set()
    for folder in all_folders:
        mode = folder.rstrip('123')
        mode_set.add(mode)

    modes = sorted(mode_set)

    mean_scores = {}
    for mode in modes:
        combined = load_runs(base_dir, mode)
        if combined is not None:
            mean_per_step = combined.groupby(combined['Step'].astype(int))['total_score'].mean()
            mean_scores[mode] = mean_per_step

    for mode, series in mean_scores.items():
        abl_type = get_ablation_type(mode, setting)
        color = ABLATION_COLORS.get(abl_type, 'black')
        display_label = ABLATION_LABELS.get(abl_type, abl_type)
        line, = ax.plot(series.index, series.values, linewidth=1.6, alpha=0.85, color=color)
        if abl_type not in legend_handles:
            legend_handles[abl_type] = (line, display_label)

    ax.set_title(SETTING_TITLES[setting], fontsize=11)
    ax.set_xlabel("Step")
    ax.set_ylabel("Mean Total Score")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Single legend at the bottom in 1 row
ordered_handles = [
    legend_handles[k] for k in ABLATION_LABEL_ORDER if k in legend_handles
]
lines = [h[0] for h in ordered_handles]
labels = [h[1] for h in ordered_handles]

fig.legend(
    lines, labels,
    fontsize=10,
    ncol=len(labels),
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    borderaxespad=0.,
    frameon=True
)

plt.tight_layout()
# plt.title("")
# plt.subplots_adjust(bottom=0.12)  # make room for the bottom legend
plt.savefig("./manuscript/figure_pdf/ablation_2x2.pdf", bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

base_dir = "./output/journal/result"

SETTINGS = ['nm', 'ns', 'sm', 'ss']
SETTING_TITLES = {
    'nm': 'Neighbor-Mask Multi-Agents (NM)',
    'ns': 'Neighbor-Mask Single-Agent (NS)',
    'sm': 'Self-Mask Multi-Agents (SM)',
    'ss': 'Self-Mask Single-Agent (SS)'
}

ABLATION_LABEL_ORDER = ['pepevolve', 'cs_topk', 'topk_gra', 'topk', 'pepinvent_cs', 'pepinvent']

ABLATION_LABELS = {
    'pepevolve': 'PepEVOLVE (CHUCKLES Shift + TopK + GRA)',
    'cs_topk':   'CHUCKLES Shift + TopK',
    'topk_gra':  'TopK + GRA',
    'topk':      'TopK',
    'pepinvent_cs': 'CHUCKLES Shift',
    'pepinvent': 'PepINVENT'
}

ABLATION_COLORS = {
    'pepevolve': "#0072B2",
    'cs_topk':   "#E69F00",
    'topk':      "#009E73",
    'topk_gra':  "#CC79A7",
    'pepinvent_cs': "#D55E00",
    'pepinvent': "#4C4C4C"
}

def get_ablation_type(mode, setting):
    if mode.startswith("pepinvent"):
        return mode
    prefix = f"{setting}_"
    if mode.startswith(prefix):
        return mode[len(prefix):]
    return mode

def load_single_run(base_dir, mode, run):
    folder = os.path.join(base_dir, f"{mode}{run}")
    if mode.startswith("pepinvent"):
        csv_path = os.path.join(folder, "results.csv")
    else:
        csv_path = os.path.join(folder, "results_evolving_step250.csv")

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['Step'] = df['Step'].astype(int)
        return df
    else:
        print(f"  ⚠️ Missing: {csv_path}")
        return None

def compute_mean_of_means(base_dir, mode):
    run_means = []

    for run in [1, 2, 3]:
        df = load_single_run(base_dir, mode, run)
        if df is not None:
            mean_per_step = df.groupby('Step')['total_score'].mean()
            run_means.append(mean_per_step)

    if not run_means:
        return None

    # Align steps safely (handles missing steps)
    aligned = pd.concat(run_means, axis=1)

    # Mean of means (equal weight per run)
    mean_of_means = aligned.mean(axis=1)

    return mean_of_means


fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

legend_handles = {}

for ax, setting in zip(axes.flat, SETTINGS):

    all_folders = [f for f in os.listdir(base_dir)
                   if os.path.isdir(os.path.join(base_dir, f))]

    all_folders = [f for f in all_folders
                   if f.startswith(setting + "_") or f.startswith("pepinvent")]

    mode_set = set()
    for folder in all_folders:
        mode = folder.rstrip('123')
        mode_set.add(mode)

    modes = sorted(mode_set)

    mean_scores = {}

    for mode in modes:
        mean_series = compute_mean_of_means(base_dir, mode)
        if mean_series is not None:
            mean_scores[mode] = mean_series

    for mode, series in mean_scores.items():
        abl_type = get_ablation_type(mode, setting)
        color = ABLATION_COLORS.get(abl_type, 'black')
        display_label = ABLATION_LABELS.get(abl_type, abl_type)

        line, = ax.plot(
            series.index,
            series.values,
            linewidth=1.6,
            alpha=0.85,
            color=color
        )

        if abl_type not in legend_handles:
            legend_handles[abl_type] = (line, display_label)

    ax.set_title(SETTING_TITLES[setting], fontsize=11)
    ax.set_xlabel("Step")
    ax.set_ylabel("Mean Total Score")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legend (single row at bottom)
ordered_handles = [
    legend_handles[k] for k in ABLATION_LABEL_ORDER if k in legend_handles
]

lines = [h[0] for h in ordered_handles]
labels = [h[1] for h in ordered_handles]

fig.legend(
    lines,
    labels,
    fontsize=10,
    ncol=len(labels),
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    borderaxespad=0.,
    frameon=True
)

plt.tight_layout()

plt.savefig(
    "./manuscript/figure_pdf/ablation_2x2.pdf",
    bbox_inches='tight',
    dpi=600
)

plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

base_dir = "./output/journal/result"

ABLATION_LABEL_ORDER = ['pepevolve', 'cs_topk', 'topk_gra', 'topk', 'pepinvent_cs', 'pepinvent']

ABLATION_LABELS = {
    'pepevolve': 'PepEVOLVE (CHUCKLES Shift + TopK + GRA)',
    'cs_topk':   'CHUCKLES Shift + TopK',
    'topk_gra':  'TopK + GRA',
    'topk':      'TopK',
    'pepinvent_cs': 'CHUCKLES Shift',
    'pepinvent': 'PepINVENT'
}

ABLATION_COLORS = {
    'pepevolve': "#0072B2",
    'cs_topk':   "#E69F00",
    'topk':      "#009E73",
    'topk_gra':  "#CC79A7",
    'pepinvent_cs': "#D55E00",
    'pepinvent': "#4C4C4C"
}

def load_runs(base_dir, mode):
    dfs = []
    for run in [1, 2, 3]:
        folder = os.path.join(base_dir, f"{mode}{run}")
        csv_path = os.path.join(folder, "results.csv" if mode.startswith("pepinvent") else "results_evolving_step250.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df['run'] = run
            dfs.append(df)
        else:
            print(f"  ⚠️ Missing: {csv_path}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

def get_unique_smiles_avg_score(combined):
    return combined.groupby('SMILES')['total_score'].mean()

def get_ablation_type(mode, setting):
    if mode.startswith("pepinvent"):
        return mode
    prefix = f"{setting}_"
    if mode.startswith(prefix):
        return mode[len(prefix):]
    return mode

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)

legend_handles = {}

for ax, setting in zip(axes.flat, SETTINGS):
    all_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
    all_folders = [f for f in all_folders if f.startswith(setting + "_") or f.startswith("pepinvent")]

    mode_set = set()
    for folder in all_folders:
        mode = folder.rstrip('123')
        mode_set.add(mode)

    modes = sorted(mode_set)

    for mode in modes:
        combined = load_runs(base_dir, mode)
        if combined is None:
            continue
        abl_type = get_ablation_type(mode, setting)
        color = ABLATION_COLORS.get(abl_type, 'black')
        display_label = ABLATION_LABELS.get(abl_type, abl_type)
        avg_scores = get_unique_smiles_avg_score(combined)
        n, bins_, patches = ax.hist(avg_scores, bins=50, alpha=0.5, color=color, density=True)
        if abl_type not in legend_handles:
            legend_handles[abl_type] = (patches[0], display_label)

    ax.set_title(SETTING_TITLES[setting], fontsize=11)
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Single legend at the bottom in 1 row
ordered_handles = [
    legend_handles[k] for k in ABLATION_LABEL_ORDER if k in legend_handles
]
patches = [h[0] for h in ordered_handles]
labels = [h[1] for h in ordered_handles]

fig.legend(
    patches, labels,
    fontsize=10,
    ncol=len(labels),
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    borderaxespad=0.,
    frameon=True
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.savefig("./manuscript/figure_pdf/ablation_score_dist_2x2.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np

base_dir = "./output/journal/result"

SETTINGS = ['nm', 'ns', 'sm', 'ss']
ABLATION_TYPES = ['pepevolve', 'cs_topk', 'topk', 'topk_gra', 'pepinvent', 'pepinvent_cs']

def load_runs(base_dir, mode):
    dfs = []
    for run in [1, 2, 3]:
        folder = os.path.join(base_dir, f"{mode}{run}")
        csv_path = os.path.join(folder, "results.csv" if mode.startswith("pepinvent") else "results_evolving_step250.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df['run'] = run
            dfs.append(df)
        else:
            print(f"  ⚠️ Missing: {csv_path}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

def get_unique_smiles_avg_score(combined):
    return combined.groupby('SMILES')['total_score'].mean()

def get_ablation_type(mode, setting):
    if mode.startswith("pepinvent"):
        return mode
    prefix = f"{setting}_"
    if mode.startswith(prefix):
        return mode[len(prefix):]
    return mode

# ── Collect scores ────────────────────────────────────────────────────────────
results = {abl: {} for abl in ABLATION_TYPES}

for setting in SETTINGS:
    all_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
    setting_folders = [f for f in all_folders if f.startswith(setting + "_")]
    mode_set = set(f.rstrip('123') for f in setting_folders)

    for mode in sorted(mode_set):
        combined = load_runs(base_dir, mode)
        if combined is None:
            continue
        abl_type = get_ablation_type(mode, setting)
        avg_scores = get_unique_smiles_avg_score(combined).sort_values(ascending=False)
        results[abl_type][setting] = avg_scores

for pi_mode in ['pepinvent', 'pepinvent_cs']:
    combined = load_runs(base_dir, pi_mode)
    if combined is not None:
        avg_scores = get_unique_smiles_avg_score(combined).sort_values(ascending=False)
        results[pi_mode]['_global'] = avg_scores

# ── Build tables ──────────────────────────────────────────────────────────────
def get_topk_mean_std(scores, k):
    if scores is None:
        return np.nan, np.nan
    top = scores.iloc[:k] if len(scores) >= k else scores
    return round(top.mean(), 4), round(top.std(), 4)

def build_topk_table_numeric(results, k):
    """Returns a DataFrame with numeric mean values (for finding max per column)."""
    setting_ablations = [a for a in ABLATION_TYPES if not a.startswith("pepinvent")]
    pepinvent_ablations = [a for a in ABLATION_TYPES if a.startswith("pepinvent")]
    rows = []

    for abl in setting_ablations:
        row = {'ablation': abl}
        for setting in SETTINGS:
            scores = results[abl].get(setting)
            mean, std = get_topk_mean_std(scores, k)
            row[f"{setting}_mean"] = mean
            row[f"{setting}_std"] = std
        rows.append(row)

    for abl in pepinvent_ablations:
        scores = results[abl].get('_global')
        mean, std = get_topk_mean_std(scores, k)
        row = {'ablation': abl}
        for setting in SETTINGS:
            row[f"{setting}_mean"] = mean
            row[f"{setting}_std"] = std
        rows.append(row)

    return pd.DataFrame(rows).set_index('ablation')

BOLD_GREEN  = "\033[1;32m"
BOLD_YELLOW = "\033[1;33m"
RESET       = "\033[0m"

def print_topk_table_colored(results, k):
    df = build_topk_table_numeric(results, k)

    # Find best and second-best (max mean) row index per setting column
    best_per_setting = {}
    second_per_setting = {}
    for setting in SETTINGS:
        col = df[f"{setting}_mean"].dropna()
        sorted_idx = col.sort_values(ascending=False).index
        best_per_setting[setting] = sorted_idx[0] if len(sorted_idx) >= 1 else None
        second_per_setting[setting] = sorted_idx[1] if len(sorted_idx) >= 2 else None

    col_width = 22
    header = f"{'ablation':<15}" + "".join(f"{s:>{col_width}}" for s in SETTINGS)
    print(header)
    print("-" * (15 + col_width * len(SETTINGS)))

    for abl in df.index:
        row_str = f"{abl:<15}"
        for setting in SETTINGS:
            mean = df.loc[abl, f"{setting}_mean"]
            std  = df.loc[abl, f"{setting}_std"]
            if np.isnan(mean):
                cell = "N/A"
            else:
                cell = f"{mean:.4f} ± {std:.4f}"
            # Highlight best in green, second-best in yellow
            if abl == best_per_setting[setting]:
                cell = f"{BOLD_GREEN}{cell}{RESET}"
                row_str += f"{cell:>{col_width + len(BOLD_GREEN) + len(RESET)}}"
            elif abl == second_per_setting[setting]:
                cell = f"{BOLD_YELLOW}{cell}{RESET}"
                row_str += f"{cell:>{col_width + len(BOLD_YELLOW) + len(RESET)}}"
            else:
                row_str += f"{cell:>{col_width}}"
        print(row_str)

print("=" * 70)
print(f"TOP-1 MEAN ± STD SCORE")
print("=" * 70)
print_topk_table_colored(results, k=1)

print("\n" + "=" * 70)
print(f"TOP-10 MEAN ± STD SCORE")
print("=" * 70)
print_topk_table_colored(results, k=10)

# 6. (Average runs) PepEVOLVE vs PepINVENT

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

base_dir = "./output/journal/result"

def compute_mean_of_means(base_dir, mode):
    run_means = []

    for run in [1, 2, 3]:
        df = load_single_run(base_dir, mode, run)
        if df is not None:
            mean_per_step = df.groupby('Step')['total_score'].mean()
            run_means.append(mean_per_step)

    if not run_means:
        return None

    # Align steps safely (handles missing steps)
    aligned = pd.concat(run_means, axis=1)

    # Mean of means (equal weight per run)
    mean_of_means = aligned.mean(axis=1)

    return mean_of_means

SETTINGS = ['nm', 'ns', 'sm', 'ss']
SETTING_TITLES = {
    'nm': 'NM',
    'ns': 'NS',
    'sm': 'SM',
    'ss': 'SS'
}

PEPEVOLVE_COLORS = {
    'nm': 'tab:blue',
    'ns': 'tab:green',
    'sm': 'tab:purple',
    'ss': 'tab:red',
}
PEPINVENT_COLOR = 'tab:gray'

fig, ax = plt.subplots(figsize=(5, 4))

legend_handles = {}

# Plot PepEVOLVE for each setting
for setting in SETTINGS:
    mode = f"{setting}_pepevolve"
    mean_series = compute_mean_of_means(base_dir, mode)
    if mean_series is not None:
        label = f"PepEVOLVE - {SETTING_TITLES[setting].upper()}"
        line, = ax.plot(
            mean_series.index,
            mean_series.values,
            linewidth=1.6,
            alpha=0.8,
            color=PEPEVOLVE_COLORS[setting],
            label=label
        )
        legend_handles[f"pepevolve_{setting}"] = (line, label)

# Plot PepINVENT (single, shared across settings)
mean_series_pepinvent = compute_mean_of_means(base_dir, "pepinvent")
if mean_series_pepinvent is not None:
    label = "PepINVENT"
    line, = ax.plot(
        mean_series_pepinvent.index,
        mean_series_pepinvent.values,
        linewidth=1.6,
        alpha=0.8,
        color=PEPINVENT_COLOR,
        label=label
    )
    legend_handles["pepinvent"] = (line, label)

ax.set_title("PepEVOLVE vs PepINVENT across Settings", fontsize=12)
ax.set_xlabel("Step")
ax.set_ylabel("Mean Total Score")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ordered_keys = [f"pepevolve_{s}" for s in SETTINGS] + ["pepinvent"]
ordered_handles = [legend_handles[k] for k in ordered_keys if k in legend_handles]
lines = [h[0] for h in ordered_handles]
labels = [h[1] for h in ordered_handles]

ax.legend(
    lines,
    labels,
    fontsize=10,
    ncol=1,
    loc='upper left',
    frameon=True
)

plt.tight_layout()

plt.savefig(
    "./figure_pdf/average_score_linegraph_all_raw.pdf",
    bbox_inches='tight',
    dpi=600
)

plt.show()

In [ ]:
PEPEVOLVE_COLORS = {
    'nm': 'tab:blue',
    'ns': 'tab:green',
    'sm': 'tab:purple',
    'ss': 'tab:red',
}
PEPINVENT_COLOR = 'tab:gray'

SETTING_LABELS = {
    'nm': 'PepEVOLVE - NM',
    'ns': 'PepEVOLVE - NS',
    'sm': 'PepEVOLVE - SM',
    'ss': 'PepEVOLVE - SS',
}

def load_runs(base_dir, mode):
    dfs = []
    for run in [1, 2, 3]:
        folder = os.path.join(base_dir, f"{mode}{run}")
        csv_path = os.path.join(folder, "results.csv" if mode.startswith("pepinvent") else "results_evolving_step250.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df['run'] = run
            dfs.append(df)
        else:
            print(f"  ⚠️ Missing: {csv_path}")
    return pd.concat(dfs, ignore_index=True) if dfs else None

def get_unique_smiles_avg_score(combined):
    return combined.groupby('SMILES')['total_score'].mean()

fig, ax = plt.subplots(figsize=(6, 4))

legend_handles = {}

for setting in SETTINGS:
    mode = f"{setting}_pepevolve"
    combined = load_runs(base_dir, mode)
    if combined is None:
        continue
    color = PEPEVOLVE_COLORS[setting]
    label = SETTING_LABELS[setting]
    avg_scores = get_unique_smiles_avg_score(combined)
    n, bins_, patches = ax.hist(avg_scores, bins=100, alpha=0.7, color=color, density=True)
    legend_handles[f'pepevolve_{setting}'] = (patches[0], label)

combined_pepinvent = load_runs(base_dir, "pepinvent")
if combined_pepinvent is not None:
    avg_scores = get_unique_smiles_avg_score(combined_pepinvent)
    n, bins_, patches = ax.hist(avg_scores, bins=100, alpha=0.7, color=PEPINVENT_COLOR, density=True)
    legend_handles['pepinvent'] = (patches[0], 'PepINVENT')

ax.set_xlabel("Score")
ax.set_ylabel("Density")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ordered_keys = [f'pepevolve_{s}' for s in SETTINGS] + ['pepinvent']
ordered_handles = [legend_handles[k] for k in ordered_keys if k in legend_handles]
patch_list = [h[0] for h in ordered_handles]
label_list = [h[1] for h in ordered_handles]

ax.legend(
    patch_list, label_list,
    fontsize=10,
    loc='upper left',
    frameon=True
)

plt.tight_layout()
plt.savefig("./figure_pdf/average_score_distribution.pdf", bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# Bins and labels
bins = np.array([0.4, 0.6, 0.8, 0.9, 0.94, 1.0])
labels = [f"{bins[i]:.2f}–{bins[i+1]:.2f}" for i in range(len(bins)-1)]
n_bins = len(labels)

def get_unique_avg_scores(base_dir, mode):
    combined = load_runs(base_dir, mode)
    if combined is None:
        return None
    return combined.groupby('SMILES')['total_score'].mean()

# Load unique SMILES with avg scores across 3 runs
pepinvent_scores      = get_unique_avg_scores(base_dir, 'pepinvent')
self_multi_scores     = get_unique_avg_scores(base_dir, 'sm_pepevolve')
self_single_scores    = get_unique_avg_scores(base_dir, 'ss_pepevolve')
neighbor_single_scores = get_unique_avg_scores(base_dir, 'ns_pepevolve')
neighbor_multi_scores  = get_unique_avg_scores(base_dir, 'nm_pepevolve')

# Histograms per dataset
pepinvent_hist, _        = np.histogram(pepinvent_scores,       bins=bins)
self_multi_hist, _       = np.histogram(self_multi_scores,      bins=bins)
self_single_hist, _      = np.histogram(self_single_scores,     bins=bins)
neighbor_single_hist, _  = np.histogram(neighbor_single_scores, bins=bins)
neighbor_multi_hist, _   = np.histogram(neighbor_multi_scores,  bins=bins)

# Stack the data
data = np.vstack([
    pepinvent_hist,
    self_multi_hist,
    self_single_hist,
    neighbor_single_hist,
    neighbor_multi_hist
])

colors_stack = ['#d73027', '#fc8d59', '#fee08b', '#91bfdb', '#a2fab1']

datasets = ['PepINVENT', 'PepEVOLVE - SM', 'PepEVOLVE - SS', 'PepEVOLVE - NS', 'PepEVOLVE - NM']
x_pos = np.arange(len(datasets))
bottom = np.zeros(len(datasets), dtype=int)

fig, ax = plt.subplots(figsize=(8, 4))

for i in range(n_bins):
    ax.bar(
        x_pos, data[:, i],
        bottom=bottom,
        label=labels[i],
        color=colors_stack[i],
        edgecolor='black',
        linewidth=0.4
    )
    bottom += data[:, i]

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('Number of Unique Peptides', fontsize=11)
ax.set_xticks(x_pos, datasets, rotation=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(title='Score Range', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.savefig(
    "./figure_pdf/average_score_stacked_bar.pdf",
    dpi=600, bbox_inches='tight'
)
plt.show()